In [2]:
import os
import json
import pandas as pd
from pathlib import Path

data_dir = Path("../data/raw/nhanes")
output_file = data_dir / "_description.json"

def generate_schema_skeleton():
    if not data_dir.exists():
        print(f"[ERRO] O diretório {data_dir.resolve()} não foi encontrado.")
        return

    json_structure = []

    xpt_files = list(data_dir.glob("*.xpt"))
    print(f"Encontrados {len(xpt_files)} arquivos .xpt. Iniciando extração de metadados...\n")

    for file_path in xpt_files:
        file_name = file_path.name

        try:
            reader = pd.read_sas(file_path, format='xport', iterator=True, chunksize=1)
            first_chunk = next(reader)
            columns = first_chunk.columns.tolist()

            reader.close()

            columns_list = [
                {
                    "column_name": col,
                    "presentation_name": "",
                    "presentation_description": ""
                }
                for col in columns
            ]

            file_obj = {
                "file": {
                    "file_name": file_name,
                    "presentation_name": "",
                    "presentation_description": "",
                    "columns": columns_list
                }
            }

            json_structure.append(file_obj)
            print(f"[OK] Arquivo mapeado: {file_name} ({len(columns)} colunas extraídas).")

        except Exception as e:
            print(f"[ERRO] Falha ao processar o arquivo {file_name}. Detalhe: {e}")

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(json_structure, f, indent=2, ensure_ascii=False)

    print(f"\nProcesso concluído com sucesso!")
    print(f"O arquivo de configuração foi gerado em: {output_file.resolve()}")

generate_schema_skeleton()

Encontrados 74 arquivos .xpt. Iniciando extração de metadados...

[OK] Arquivo mapeado: ACQ_L.xpt (5 colunas extraídas).
[OK] Arquivo mapeado: AGP_L.xpt (3 colunas extraídas).
[OK] Arquivo mapeado: ALB_CR_L.xpt (8 colunas extraídas).
[OK] Arquivo mapeado: ALQ_L.xpt (9 colunas extraídas).
[OK] Arquivo mapeado: AUQ_L.xpt (14 colunas extraídas).
[OK] Arquivo mapeado: BAQ_L.xpt (15 colunas extraídas).
[OK] Arquivo mapeado: BAX_L.xpt (45 colunas extraídas).
[OK] Arquivo mapeado: BIOPRO_L.xpt (42 colunas extraídas).
[OK] Arquivo mapeado: BMX_L.xpt (22 colunas extraídas).
[OK] Arquivo mapeado: BPQ_L.xpt (6 colunas extraídas).
[OK] Arquivo mapeado: BPXO_L.xpt (12 colunas extraídas).
[OK] Arquivo mapeado: CBC_L.xpt (23 colunas extraídas).
[OK] Arquivo mapeado: DBQ_L.xpt (27 colunas extraídas).
[OK] Arquivo mapeado: DEMO_L.xpt (27 colunas extraídas).
[OK] Arquivo mapeado: DEQ_L.xpt (4 colunas extraídas).
[OK] Arquivo mapeado: DIQ_L.xpt (9 colunas extraídas).
[OK] Arquivo mapeado: DPQ_L.xpt (11 c

In [5]:
import os
import json
import requests
from bs4 import BeautifulSoup
import time
from pathlib import Path

# 1. Configuração de Diretórios Dinâmicos
data_dir = Path("../data/raw/nhanes")
output_file = "../data/raw/nhanes/_nhanes_descriptions.json"

# 2. Padrão Fallback de Rotas (Análogo a uma lista de Endpoints no Java/Spring)
anos_busca = ["2021", "1999"]
base_url_template = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{ano}/DataFiles/{base_name}.htm"

# 3. Validação do Diretório (Fail-Fast)
if not data_dir.exists():
    raise FileNotFoundError(f"[ERRO ARQUITETURAL] O diretório {data_dir.resolve()} não existe. Verifique o caminho.")

file_list = [f.name for f in data_dir.glob("*.xpt")]

if not file_list:
    raise ValueError(f"[AVISO] Nenhum arquivo .xpt foi encontrado em {data_dir.resolve()}.")

json_structure = []
erros = []

print(f"Encontrados {len(file_list)} arquivos .xpt em {data_dir.resolve()}.")
print("Iniciando o Web Scraping com motor de Fallback (2021 -> 1999) no portal do CDC...\n")

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

for file_name in file_list:
    base_name = file_name.replace(".xpt", "")

    file_obj = {
        "file": {
            "file_name": file_name,
            "presentation_name": "",
            "presentation_description": "",
            "columns": []
        }
    }

    sucesso_scraping = False

    # Chain of Responsibility (Itera sobre a lista de anos até ter sucesso)
    for ano in anos_busca:
        url = base_url_template.format(ano=ano, base_name=base_name)

        try:
            response = requests.get(url, headers=headers, timeout=10)

            # Se encontrou a página, para de tentar outros anos
            if response.status_code == 200:
                soup = BeautifulSoup(response.text, 'html.parser')

                title_tag = soup.find('h1')
                if title_tag:
                    file_obj["file"]["presentation_name"] = title_tag.text.strip()

                codebook_ul = soup.find('ul', id='CodebookLinks')

                if codebook_ul:
                    for li in codebook_ul.find_all('li'):
                        a_tag = li.find('a')
                        if a_tag:
                            text = a_tag.text.strip()

                            if " - " in text:
                                col_name, desc = text.split(" - ", 1)
                            else:
                                col_name = text
                                desc = ""

                            file_obj["file"]["columns"].append({
                                "column_name": col_name.strip(),
                                "presentation_name": desc.strip(),
                                "presentation_description": ""
                            })
                    print(f"[OK] Scraping concluído: {file_name} (Rota: {ano})")
                    sucesso_scraping = True
                    break # Interrompe o loop de anos, pois já extraiu os dados com sucesso
                else:
                    # A página existe, mas não tem o layout esperado
                    pass

            elif response.status_code == 404:
                # O arquivo não existe neste ano, deixa o loop prosseguir para tentar o Fallback
                continue

        except Exception as e:
            # Em caso de timeout da rede, deixa tentar o próximo ano
            pass

        time.sleep(0.5) # Rate limiting

    # Após esgotar todos os anos (2021 e 1999) e não conseguir fazer o scraping:
    if not sucesso_scraping:
        print(f"[ERRO] Falha ao extrair metadados para: {file_name} (Todas as rotas de Fallback esgotadas)")
        erros.append(file_name)

    json_structure.append(file_obj)

# 4. Serialização
os.makedirs(os.path.dirname(output_file), exist_ok=True)
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(json_structure, f, indent=2, ensure_ascii=False)

print("\n=== RESUMO DA EXECUÇÃO ===")
print(f"Arquivo gerado em: {os.path.abspath(output_file)}")
print(f"Total de arquivos processados com sucesso: {len(file_list) - len(erros)}")
if erros:
    print(f"Arquivos que apresentaram falha ou layout HTML não suportado: {erros}")

Encontrados 74 arquivos .xpt em D:\projects\tech-challenge-11iadt-fase-01\data\raw\nhanes.
Iniciando o Web Scraping com motor de Fallback (2021 -> 1999) no portal do CDC...

[OK] Scraping concluído: ACQ_L.xpt (Rota: 2021)
[OK] Scraping concluído: AGP_L.xpt (Rota: 2021)
[OK] Scraping concluído: ALB_CR_L.xpt (Rota: 2021)
[OK] Scraping concluído: ALQ_L.xpt (Rota: 2021)
[OK] Scraping concluído: AUQ_L.xpt (Rota: 2021)
[OK] Scraping concluído: BAQ_L.xpt (Rota: 2021)
[OK] Scraping concluído: BAX_L.xpt (Rota: 2021)
[OK] Scraping concluído: BIOPRO_L.xpt (Rota: 2021)
[OK] Scraping concluído: BMX_L.xpt (Rota: 2021)
[OK] Scraping concluído: BPQ_L.xpt (Rota: 2021)
[OK] Scraping concluído: BPXO_L.xpt (Rota: 2021)
[OK] Scraping concluído: CBC_L.xpt (Rota: 2021)
[OK] Scraping concluído: DBQ_L.xpt (Rota: 2021)
[OK] Scraping concluído: DEMO_L.xpt (Rota: 2021)
[OK] Scraping concluído: DEQ_L.xpt (Rota: 2021)
[OK] Scraping concluído: DIQ_L.xpt (Rota: 2021)
[OK] Scraping concluído: DPQ_L.xpt (Rota: 2021)
[O